## Merge Author Profiles — Anchor + Stragglers (Phase 5)

**Phase 5** — name-rarity + works-count-imbalance merge. The shape "one substantial anchor profile + 1–4 small straggler profiles, all sharing a parsed name" identifies splinters where the works distribution is so skewed that the dominant profile has effectively claimed the name. Catches the Jason Priem + Kyle Demes residual-splinter classes that earlier phases missed.

**Why this rule next:** the existing portfolio (Phases 1–4b/c) all rely on inst-overlap or coauthor-overlap as the primary precision signal. This phase adds an orthogonal axis — the anchor's works-count dominance is itself the precision signal. Catches the residual no-ORCID-anchor cohort that Phase 4 / Phase 4b/c excluded, and adds a (first, last)-only Demes class for middle-initial-vs-no-middle splinters that don't share a parsed name.

**Locked SHIP envelope (option A) — 6 sub-rules:**

| # | Recipe | Sizes | Guard | Splinters | Audit precision |
|---|---|---|---|---|---|
| 1 | 2 profiles · anchor ≥30 works · splinter 1-5 | size 2 | none | 480K | 97% (n=55) |
| 2 | 2 profiles · anchor 10-29 works · splinter 1-5 | size 2 | none | 697K | 96% (n=25) |
| 3 | 4 profiles · anchor ≥30 works · splinter 1-5 each | size 4 | none | 369K | 95% (n=72) |
| 4 | 5 profiles · anchor ≥30 works · splinter 1-5 each | size 5 | + shared topic | 175K | ~93% (n=76) |
| 5 | 3 profiles · anchor ≥30 works · splinter 1-5 each | size 3 | + shared topic | 256K | ~96% (n=62) |
| 6 | 2 profiles · same first/last · anchor mid-initial + splinter no-middle · anchor ≥10 · splinter 1-5 (Demes class) | size 2 | + year-window overlap (≤10 yr) | ~120K | ~92-95% (n=25) |
| | **Total** | | | **~2.10M splinters across ~1.55M anchors** | **~94% blended** |

**Cumulative FP estimate:** ~91K wrongly-merged splinters across 2.10M = **~4.3% blended FP rate**, validated across n=315 hand-judged audit pairs.

**Comparison with shipped portfolio:**

| Phase | Splinters | Audit precision | Cumulative FPs |
|---|---|---|---|
| 1 (ORCID collision) | 191K | 99.5% | ~1K |
| 2 (rich-name) | 400K | 96% | ~16K |
| 3b (one-ORCID-anchor) | 101K | 99% | ~1K |
| 3c (rich both-no-ORCID) | 88K | 96% | ~3.5K |
| 4 (no-middle anchor INST+COA) | 103K | 98% | ~2K |
| 4b/c (no-middle COA-only) | 160K | 98% | ~3.2K |
| **Phase 5 (this notebook)** | **~2.10M** | **~94%** | **~91K** |

Phase 5 is roughly 2× the volume of all 6 prior phases combined, at slightly higher per-merge FP rate.

**Key precision guards (per recipe row):**

1. **Length-3 first floor.** `LENGTH(REGEXP_REPLACE(p_first, '[.]', '')) >= 3`. Drops initials-first cases like 'a', 's.', 'sj' that conflate distinct humans.
2. **Length-2 last floor.** Standard guard from Phases 3c/4.
3. **Dominant raw_name majority ≥ 50%** per profile. Per Phase 2/3c/4 — protects against parser-pollution stub raw names dominating a profile's parsed key.
4. **Full_name self-consistency.** `p_last` must appear as a whole token in the profile's `full_name`. Per Phase 3b/c. Catches the Vieira-class issue (full_name disagrees with dominant raw).
5. **Works-count imbalance.** All splinters in [1, 5] works AND anchor ≥ 10 works. The load-bearing precision signal.
6. **Sink cap.** `anchor_works <= 5,000`. Per all earlier phases.
7. **Length-3 first / length-2 last** AND no internal periods in first.

**Per-row additional guards:**
- Row 4 (size 5) AND Row 5 (size 3): require `ARRAYS_OVERLAP(anchor.topic_ids, straggler.topic_ids)` per pair. Lifts these rows from 93-94% to ~96-100% projected.
- Row 6 (Demes class): groups by (first, last) only ignoring middle, but requires anchor's middle to be initial (1-2 chars) AND splinter's middle empty. Year-window overlap guard: `straggler.last_year >= anchor.first_year - 10 AND straggler.first_year <= anchor.last_year + 10`.

**Winner selection:** anchor = profile with `MAX(works_count)` per group. Tiebreaker on equal max: `MIN(created_date)` → `MIN(author_id)`. Same convention as Phase 3c.

**Loser handling:** *No DELETE.* MERGE-then-soft-zero-via-CreateAuthors approach (same as Phase 1 reversal onward).

**Important: net-of-merge-logs uses LOSER-only filter.** Past winners must remain candidates so they can absorb additional splinters. Excluding winners (an earlier bug) lost ~170K splinter merges including the Jason Priem residual — now fixed.

**Rollback plan:** Audit log `openalex.authors.author_merge_log_anchor_stragglers` captures anchor and loser metadata + recipe row + signal flags; sufficient to reconstruct any loser via `STILL_UNMATCHED` + `MatchAuthors`.

**Known residual gaps (out of scope for Phase 5):**
- **Anchor 10-29 with size 3+** (88-91% precision in audit) — would need extra signal to clear 95% bar.
- **Anchor 30+ with size 7+** (CJK character-variant collapse; 90% → 78% → 55% as size grows) — structural ceiling, no guard recovers cleanly.
- **Anchor < 10 works** (73% precision; can't establish dominance) — entirely off-limits.
- **Rich-middle-vs-no-middle splinters** — not currently in Demes class (which is initial-only). Would need a Phase 5b extension.

### Cell 1: Build merge candidates table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.merge_candidates_anchor_stragglers AS
WITH
all_loser_ids AS (
  -- Only past LOSERS, not winners. Past winners must remain candidates so they can
  -- absorb additional splinters. Excluding winners loses the Priem residual splinter.
  SELECT loser_author_id AS aid FROM openalex.authors.author_merge_log_orcid
  UNION SELECT loser_author_id FROM openalex.authors.author_merge_log_richname
  UNION SELECT loser_author_id FROM openalex.authors.author_merge_log_orcid_anchor_multi
  UNION SELECT loser_author_id FROM openalex.authors.author_merge_log_richname_multi
  UNION SELECT loser_author_id FROM openalex.authors.author_merge_log_no_middle_anchor
  UNION SELECT loser_author_id FROM openalex.authors.author_merge_log_coauthor_anchor
),
all_profiles AS (
  SELECT id AS author_id, orcid, full_name, display_name, block_key,
         works_count, cited_by_count, created_date,
         TRANSFORM(COALESCE(last_known_institutions, ARRAY()), x -> x.id) AS lki_ids,
         TRANSFORM(COALESCE(affiliations,            ARRAY()), x -> x.institution.id) AS aff_ids,
         TRANSFORM(COALESCE(topics,                  ARRAY()), x -> x.id) AS topic_ids,
         -- First/last active year derived from counts_by_year (works_count > 0 only)
         ARRAY_MIN(TRANSFORM(
           FILTER(COALESCE(counts_by_year, ARRAY()), x -> x.works_count > 0),
           x -> x.year
         )) AS first_active_year,
         ARRAY_MAX(TRANSFORM(
           FILTER(COALESCE(counts_by_year, ARRAY()), x -> x.works_count > 0),
           x -> x.year
         )) AS last_active_year
  FROM openalex.authors.openalex_authors
  WHERE works_count > 0 AND works_count <= 5000
),
profiles_net AS (
  SELECT p.* FROM all_profiles p
  LEFT ANTI JOIN all_loser_ids ll ON p.author_id = ll.aid
),
ra_counts AS (
  SELECT wa.author_id, wa.raw_author_name, COUNT(*) AS n_works
  FROM openalex.works.work_authors wa
  JOIN profiles_net n ON n.author_id = wa.author_id
  WHERE wa.raw_author_name IS NOT NULL AND TRIM(wa.raw_author_name) <> ''
  GROUP BY wa.author_id, wa.raw_author_name
),
ra_dominant AS (
  -- Dominant raw_name per profile, weighted by work_authors row count.
  -- Require >= 50% majority (Phase 2/3/4 recipe).
  SELECT author_id, raw_author_name
  FROM (
    SELECT author_id, raw_author_name, n_works,
           SUM(n_works) OVER (PARTITION BY author_id) AS total_works,
           ROW_NUMBER() OVER (PARTITION BY author_id
                              ORDER BY n_works DESC, raw_author_name ASC) AS rn
    FROM ra_counts
  )
  WHERE rn = 1 AND n_works * 2 >= total_works
),
parsed_dominant AS (
  -- Resolve dominant raw_name to (first, middle, last). Phase 5 filters:
  --  * first >= 3 chars after stripping any periods (drops initials-first like 'a', 's.', 'sj')
  --  * last >= 2 chars after stripping trailing period
  --  * NO restriction on middle shape (Phase 5 covers rich, no-middle, AND initial middles)
  SELECT rd.author_id,
         REGEXP_REPLACE(an.parsed_name.first,  '[.]$', '') AS p_first,
         COALESCE(REGEXP_REPLACE(an.parsed_name.middle, '[.]$', ''), '') AS p_middle,
         REGEXP_REPLACE(an.parsed_name.last,   '[.]$', '') AS p_last
  FROM ra_dominant rd
  JOIN openalex.authors.author_names an ON an.raw_author_name = rd.raw_author_name
  WHERE LENGTH(REGEXP_REPLACE(an.parsed_name.first, '[.]', '')) >= 3
    AND LENGTH(REGEXP_REPLACE(an.parsed_name.last,  '[.]$', '')) >= 2
),
profile_with_dom AS (
  SELECT n.*, p.p_first, p.p_middle, p.p_last,
         CASE
           WHEN p.p_middle = '' OR p.p_middle IS NULL THEN 'no_middle'
           WHEN LENGTH(REGEXP_REPLACE(p.p_middle, '[.]', '')) <= 2 THEN 'initial_middle'
           ELSE 'rich_middle'
         END AS middle_shape,
         -- full_name self-consistency: p_last must appear as a whole token in full_name
         CASE
           WHEN n.full_name IS NULL THEN FALSE
           WHEN INSTR(
             CONCAT(' ', LOWER(REGEXP_REPLACE(
               TRANSLATE(n.full_name,
                 'áéíóúüñçÁÉÍÓÚÜÑÇàèìòùâêîôûÀÈÌÒÙÂÊÎÔÛãõÃÕıİğĞşŞçÇöÖüÜæøłÆØŁčćžšŠ',
                 'aeiouuncAEIOUUNCaeiouaeiouAEIOUAEIOUaoaoiIgGsScCoOuUaolAOLccszS'),
               '[^a-zA-Z]', ' ')), ' '),
             CONCAT(' ', p.p_last, ' ')
           ) > 0 THEN TRUE
           ELSE FALSE
         END AS full_name_ok
  FROM profiles_net   n
  JOIN parsed_dominant p ON p.author_id = n.author_id
),
-- ============================================================================
-- Recipe rows 1-5: same parsed-name (first, middle, last) match
-- ============================================================================
same_parsed_groups AS (
  SELECT p_first, p_middle, p_last,
    COUNT(*) AS n_in_key,
    MAX(works_count) AS anchor_works,
    ARRAY_MAX(SLICE(SORT_ARRAY(COLLECT_LIST(works_count), false), 2, 100)) AS max_straggler
  FROM profile_with_dom
  WHERE full_name_ok
  GROUP BY p_first, p_middle, p_last
),
ship_same_parsed_groups AS (
  SELECT p_first, p_middle, p_last, n_in_key, anchor_works, max_straggler,
    CASE
      WHEN n_in_key = 2 AND anchor_works BETWEEN 10 AND 29 THEN 'row2_modest_sz2'
      WHEN n_in_key = 2 AND anchor_works >= 30 THEN 'row1_strong_sz2'
      WHEN n_in_key = 3 AND anchor_works >= 30 THEN 'row5_strong_sz3_topic'
      WHEN n_in_key = 4 AND anchor_works >= 30 THEN 'row3_strong_sz4'
      WHEN n_in_key = 5 AND anchor_works >= 30 THEN 'row4_strong_sz5_topic'
    END AS recipe_row
  FROM same_parsed_groups
  WHERE max_straggler BETWEEN 1 AND 5
    AND (
      (n_in_key = 2 AND anchor_works BETWEEN 10 AND 29)
      OR (n_in_key BETWEEN 2 AND 5 AND anchor_works >= 30)
    )
),
same_parsed_anchor AS (
  SELECT g.p_first, g.p_middle, g.p_last, g.n_in_key, g.recipe_row,
    p.author_id AS anchor_id,
    p.full_name AS anchor_full_name,
    p.orcid AS anchor_orcid,
    p.works_count AS anchor_works_count,
    p.cited_by_count AS anchor_cited_by_count,
    p.created_date AS anchor_created_date,
    p.first_active_year AS anchor_first_year,
    p.last_active_year AS anchor_last_year,
    p.topic_ids AS anchor_topic_ids,
    p.block_key AS anchor_block_key,
    ROW_NUMBER() OVER (PARTITION BY g.p_first, g.p_middle, g.p_last
                       ORDER BY p.works_count DESC, p.created_date ASC, p.author_id ASC) AS anchor_rn
  FROM ship_same_parsed_groups g
  JOIN profile_with_dom p
    ON p.p_first = g.p_first AND p.p_middle = g.p_middle AND p.p_last = g.p_last
    AND p.full_name_ok
    AND p.works_count = g.anchor_works
),
same_parsed_pairs AS (
  SELECT a.recipe_row,
    a.anchor_id, a.anchor_orcid, a.anchor_full_name, a.anchor_works_count, a.anchor_cited_by_count,
    a.anchor_created_date, a.anchor_first_year, a.anchor_last_year, a.anchor_topic_ids, a.anchor_block_key,
    a.p_first, a.p_middle, a.p_last,
    s.author_id AS straggler_id,
    s.full_name AS straggler_full_name,
    s.orcid AS straggler_orcid,
    s.works_count AS straggler_works_count,
    s.cited_by_count AS straggler_cited_by_count,
    s.created_date AS straggler_created_date,
    s.first_active_year AS straggler_first_year,
    s.last_active_year AS straggler_last_year,
    s.topic_ids AS straggler_topic_ids,
    s.block_key AS straggler_block_key
  FROM same_parsed_anchor a
  JOIN profile_with_dom s
    ON s.p_first = a.p_first AND s.p_middle = a.p_middle AND s.p_last = a.p_last
  WHERE a.anchor_rn = 1
    AND s.author_id <> a.anchor_id
    AND s.full_name_ok
    AND s.works_count BETWEEN 1 AND 5
    -- Year-window guard (rows 1-5): straggler career must overlap anchor career within 10 years.
    -- Catches the cross-window-stub FP class (e.g., 43-yr gap Oliveira case in row 2 audit).
    -- Same predicate as row 6 (Demes class).
    AND s.first_active_year IS NOT NULL AND s.last_active_year IS NOT NULL
    AND a.anchor_first_year IS NOT NULL AND a.anchor_last_year IS NOT NULL
    AND s.last_active_year >= a.anchor_first_year - 10
    AND s.first_active_year <= a.anchor_last_year + 10
),
same_parsed_ship AS (
  -- Apply per-row guards: topic-overlap on rows 4 (size 5) and 5 (size 3)
  SELECT * FROM same_parsed_pairs
  WHERE recipe_row IN ('row1_strong_sz2', 'row2_modest_sz2', 'row3_strong_sz4')
     OR (
       recipe_row IN ('row4_strong_sz5_topic', 'row5_strong_sz3_topic')
       AND anchor_topic_ids IS NOT NULL AND straggler_topic_ids IS NOT NULL
       AND SIZE(anchor_topic_ids) > 0 AND SIZE(straggler_topic_ids) > 0
       AND ARRAYS_OVERLAP(anchor_topic_ids, straggler_topic_ids)
     )
),
-- ============================================================================
-- Recipe row 6: Demes class — same (first, last), middle differs
-- ============================================================================
demes_groups AS (
  SELECT p_first, p_last,
    COUNT(*) AS n_in_fl,
    MAX(works_count) AS anchor_works,
    ARRAY_MAX(SLICE(SORT_ARRAY(COLLECT_LIST(works_count), false), 2, 100)) AS max_straggler
  FROM profile_with_dom
  WHERE full_name_ok
  GROUP BY p_first, p_last
  HAVING COUNT(*) = 2
    AND COUNT(CASE WHEN middle_shape = 'initial_middle' THEN 1 END) = 1
    AND COUNT(CASE WHEN middle_shape = 'no_middle' THEN 1 END) = 1
    AND MAX(works_count) >= 10
    AND ARRAY_MAX(SLICE(SORT_ARRAY(COLLECT_LIST(works_count), false), 2, 100)) BETWEEN 1 AND 5
),
demes_pairs AS (
  SELECT 'row6_demes' AS recipe_row,
    a.author_id AS anchor_id, a.orcid AS anchor_orcid,
    a.full_name AS anchor_full_name, a.works_count AS anchor_works_count,
    a.cited_by_count AS anchor_cited_by_count, a.created_date AS anchor_created_date,
    a.first_active_year AS anchor_first_year, a.last_active_year AS anchor_last_year,
    a.topic_ids AS anchor_topic_ids, a.block_key AS anchor_block_key,
    a.p_first, a.p_middle, a.p_last,
    s.author_id AS straggler_id,
    s.full_name AS straggler_full_name, s.orcid AS straggler_orcid,
    s.works_count AS straggler_works_count, s.cited_by_count AS straggler_cited_by_count,
    s.created_date AS straggler_created_date,
    s.first_active_year AS straggler_first_year, s.last_active_year AS straggler_last_year,
    s.topic_ids AS straggler_topic_ids, s.block_key AS straggler_block_key
  FROM demes_groups g
  JOIN profile_with_dom a
    ON a.p_first = g.p_first AND a.p_last = g.p_last
    AND a.middle_shape = 'initial_middle' AND a.works_count = g.anchor_works
    AND a.full_name_ok
  JOIN profile_with_dom s
    ON s.p_first = g.p_first AND s.p_last = g.p_last
    AND s.middle_shape = 'no_middle' AND s.works_count <= 5
    AND s.full_name_ok
  WHERE
    -- Year-window overlap guard: straggler career within 10 years of anchor career window
    s.last_active_year IS NOT NULL AND a.first_active_year IS NOT NULL
    AND s.first_active_year IS NOT NULL AND a.last_active_year IS NOT NULL
    AND s.last_active_year >= a.first_active_year - 10
    AND s.first_active_year <= a.last_active_year + 10
)
-- ============================================================================
-- Final: union all 6 recipe rows
-- ============================================================================
SELECT * FROM same_parsed_ship
UNION ALL
SELECT * FROM demes_pairs

### Cell 2: Validation statistics

In [ ]:
%sql
SELECT recipe_row,
  COUNT(DISTINCT anchor_id) AS n_anchors,
  COUNT(*)                  AS n_splinter_pairs,
  SUM(straggler_works_count) AS works_to_rewrite,
  COUNT(CASE WHEN straggler_orcid IS NOT NULL AND straggler_orcid <> '' THEN 1 END) AS losers_with_orcid_unexpected,
  PERCENTILE_APPROX(anchor_works_count, ARRAY(0.5, 0.95, 0.99)) AS anchor_works_pctiles,
  PERCENTILE_APPROX(straggler_works_count, ARRAY(0.5, 0.95)) AS straggler_works_pctiles
FROM openalex.authors.merge_candidates_anchor_stragglers
GROUP BY recipe_row
ORDER BY recipe_row

### Cell 3: Spot-check sample (manual review) — including Priem and Demes verification

In [ ]:
%sql
-- Verify Priem and Demes residual cases are captured
SELECT recipe_row, anchor_id, anchor_full_name, anchor_works_count,
       straggler_id, straggler_full_name, straggler_works_count,
       p_first, p_middle, p_last
FROM openalex.authors.merge_candidates_anchor_stragglers
WHERE anchor_id IN (5023888391, 5086928770)
   OR straggler_id IN (5102172015, 5125160297)
ORDER BY anchor_id

In [ ]:
%sql
-- Random sample of 100 candidates for hand review
WITH sampled_anchors AS (
  SELECT DISTINCT anchor_id, recipe_row
  FROM openalex.authors.merge_candidates_anchor_stragglers
  ORDER BY RAND() LIMIT 100
)
SELECT c.recipe_row, c.anchor_id, c.anchor_orcid, c.p_first, c.p_middle, c.p_last,
       c.anchor_full_name, c.anchor_works_count, c.anchor_first_year, c.anchor_last_year,
       c.straggler_id, c.straggler_full_name,
       c.straggler_works_count, c.straggler_first_year, c.straggler_last_year
FROM openalex.authors.merge_candidates_anchor_stragglers c
JOIN sampled_anchors s ON s.anchor_id = c.anchor_id AND s.recipe_row = c.recipe_row
ORDER BY c.recipe_row, c.anchor_id, c.straggler_works_count DESC

### Cell 4: Create durable audit log

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_merge_log_anchor_stragglers AS
SELECT
  anchor_id              AS winner_author_id,
  anchor_orcid           AS winner_orcid,
  straggler_id           AS loser_author_id,
  straggler_orcid        AS loser_orcid,
  straggler_full_name    AS loser_full_name,
  straggler_works_count  AS loser_works_count,
  straggler_cited_by_count AS loser_cited_by_count,
  straggler_created_date AS loser_created_date,
  p_first, p_middle, p_last,
  recipe_row,
  anchor_works_count,
  current_timestamp()    AS merge_timestamp
FROM openalex.authors.merge_candidates_anchor_stragglers

### Cell 5: Snapshot affected work_ids (capture before MERGE rewrites author_id)

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_merge_affected_works_anchor_stragglers AS
SELECT DISTINCT wa.work_id
FROM openalex.works.work_authors wa
JOIN openalex.authors.author_merge_log_anchor_stragglers log
  ON wa.author_id = log.loser_author_id

### Cell 6: Execute the merge — rewrite work_authors.author_id from losers to winners

*No DELETE step. Phases 1 / 2 / 3b / 3c / 4 / 4b/c all use the MERGE-then-soft-zero-via-CreateAuthors approach; this notebook follows that pattern.*

In [ ]:
%sql
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT loser_author_id, winner_author_id
  FROM openalex.authors.author_merge_log_anchor_stragglers
) AS source
ON target.author_id = source.loser_author_id
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = source.winner_author_id,
    target.updated_at = current_timestamp()

### Cell 7: Queue affected works for reprocessing by UpdateWorkAuthorships

In [ ]:
%sql
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.author_merge_affected_works_anchor_stragglers

### Cell 8: Post-merge verification — losers should drain to 0 works after CreateAuthors

In [ ]:
%sql
WITH log AS (
  SELECT loser_author_id, loser_works_count, recipe_row
  FROM openalex.authors.author_merge_log_anchor_stragglers
),
loser_state AS (
  SELECT l.loser_author_id, l.recipe_row, l.loser_works_count,
         a.works_count AS current_works_count
  FROM log l
  JOIN openalex.authors.openalex_authors a ON a.id = l.loser_author_id
)
SELECT
  recipe_row,
  COUNT(*)                                                       AS total_losers,
  COUNT(CASE WHEN current_works_count = 0 THEN 1 END)            AS losers_drained_to_zero,
  COUNT(CASE WHEN current_works_count > 0 THEN 1 END)            AS losers_still_nonzero,
  PERCENTILE_APPROX(current_works_count, ARRAY(0.5, 0.95, 0.99)) AS nonzero_pctiles,
  MAX(current_works_count)                                       AS max_remaining_works
FROM loser_state
GROUP BY recipe_row
ORDER BY recipe_row